# Facial Keypoint Detection — Preprocesamiento 100k frames

Genera `images_100k.npy` y `keypoints_100k.npy` para entrenar NaimishNet.

**Requisito:** tener ya `frame_index.pkl` generado por `preprocess.ipynb`.


## 1. Imports y configuración

In [ ]:
import os
import pickle
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

# poner la ruta donde estan las images
DATA_DIR = r'F:\DL_Project\Data'
INDEX_PATH  = os.path.join(DATA_DIR, 'preprocessed', 'frame_index.pkl')
SAVE_DIR    = os.path.join(DATA_DIR, 'preprocessed')

NUM_FRAMES = 100_000
IMG_SIZE   = 96
SEED       = 42

print(f'NUM_FRAMES: {NUM_FRAMES:,}')
print(f'SAVE_DIR:   {SAVE_DIR}')


NUM_FRAMES: 100,000
SAVE_DIR:   F:\DL_Project\Data\preprocessed


## 2. Cargar índice y seleccionar 100k frames

In [ ]:
print('Cargando índice...')
with open(INDEX_PATH, 'rb') as f:
    index = pickle.load(f)
print(f'Frames totales en el índice: {len(index):,}')

# usamos la misma seed que en train.ipynb (42) para que los 50k anteriores sean un subconjunto de estos 100k
random.seed(SEED)
index_100k = random.sample(index, min(NUM_FRAMES, len(index)))
print(f'Frames seleccionados: {len(index_100k):,}')

size_gb = (NUM_FRAMES * 3 * IMG_SIZE * IMG_SIZE * 4) / 1e9
print(f'\nEspacio estimado en disco: {size_gb:.2f} GB')
print(f'RAM necesaria para cargar:  {size_gb:.2f} GB')


Cargando índice...
Frames totales en el índice: 260,399
Frames seleccionados: 100,000

Espacio estimado en disco: 11.06 GB
RAM necesaria para cargar:  11.06 GB


## 3. Preprocesar y guardar

In [3]:
images_100k    = np.zeros((NUM_FRAMES, 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
keypoints_100k = np.zeros((NUM_FRAMES, 136),                   dtype=np.float32)

for i, (npz_path, frame_idx, kps_norm) in enumerate(tqdm(index_100k, desc='Preprocesando')):
    data = np.load(npz_path, allow_pickle=True)
    img  = data['colorImages'][:, :, :, frame_idx]

    pil_img             = Image.fromarray(img)
    pil_img             = pil_img.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    images_100k[i]      = np.array(pil_img, dtype=np.float32).transpose(2, 0, 1) / 255.0
    keypoints_100k[i]   = kps_norm

print(f'\nimages_100k:    {images_100k.shape}   {images_100k.nbytes / 1e9:.2f} GB')
print(f'keypoints_100k: {keypoints_100k.shape}   {keypoints_100k.nbytes / 1e6:.1f} MB')


Preprocesando: 100%|██████████| 100000/100000 [2:39:26<00:00, 10.45it/s]   


images_100k:    (100000, 3, 96, 96)   11.06 GB
keypoints_100k: (100000, 136)   54.4 MB


## 4. Guardar en disco

In [4]:
img_path = os.path.join(SAVE_DIR, 'images_100k.npy')
kps_path = os.path.join(SAVE_DIR, 'keypoints_100k.npy')

np.save(img_path, images_100k)
np.save(kps_path, keypoints_100k)

print(f'Guardado:')
print(f'  {img_path}  ({os.path.getsize(img_path)/1e9:.2f} GB)')
print(f'  {kps_path}  ({os.path.getsize(kps_path)/1e6:.1f} MB)')
print('\n✅ Listo. Ya puedes ejecutar train_naimishnet.ipynb')


Guardado:
  F:\DL_Project\Data\preprocessed\images_100k.npy  (11.06 GB)
  F:\DL_Project\Data\preprocessed\keypoints_100k.npy  (54.4 MB)

✅ Listo. Ya puedes ejecutar train_naimishnet.ipynb
